In [14]:
from __future__ import annotations

import hashlib
import io
import json
import os
import tarfile
import time
import fnmatch
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import requests


@dataclass
class RepoLoadConfig:
    owner: str
    repo: str
    ref: str = "main"                 # branch/tag/sha
    token: Optional[str] = None       # recommended (higher limits)
    timeout_s: int = 20

    # caching
    cache_dir: Path = Path(".cache/repo_loader")
    ttl_s: int = 6 * 60 * 60          # 6 hours
    allow_stale_on_error: bool = True

    # filtering
    max_file_bytes: int = 512_000
    include_extensions: Optional[Tuple[str, ...]] = None  # e.g. (".md", ".py", ".toml")
    deny_globs: Tuple[str, ...] = (
        "**/.git/**",
        "**/.github/**",
        "**/__pycache__/**",
        "**/.venv/**",
        "**/venv/**",
        "**/node_modules/**",
        "**/dist/**",
        "**/build/**",
        "**/*.png",
        "**/*.jpg",
        "**/*.jpeg",
        "**/*.gif",
        "**/*.pdf",
        "**/*.zip",
        "**/*.tar",
        "**/*.gz",
        "**/*.7z",
        "**/*.woff",
        "**/*.woff2",
        "**/*.mp4",
        "**/*.mov",
        "**/*.mp3",
        "**/*.lock",
    )
    deny_regex: Tuple[str, ...] = (
        r".*\.env(\..*)?$",
        r".*\.pem$",
        r".*id_rsa.*",
        r".*\.key$",
        r".*\.p12$",
    )


def _headers(token: Optional[str]) -> Dict[str, str]:
    h = {"Accept": "application/vnd.github+json"}
    if token:
        h["Authorization"] = f"Bearer {token}"
    return h


def _matches_any_glob(path: str, globs: Iterable[str]) -> bool:
    p = path.replace(os.sep, "/")
    for g in globs:
        gg = g.replace(os.sep, "/")
        if fnmatch.fnmatch(p, gg) or (gg.startswith("**/") and fnmatch.fnmatch(p, gg[3:])):
            return True
    return False


def _matches_any_regex(path: str, patterns: Iterable[str]) -> bool:
    return any(re.match(pat, path) for pat in patterns)


def _is_probably_binary(data: bytes) -> bool:
    if b"\x00" in data:
        return True
    sample = data[:4096]
    if not sample:
        return False
    textish = sum(1 for b in sample if 9 <= b <= 13 or 32 <= b <= 126)
    return textish < (len(sample) * 0.80)


def _cache_key(cfg: RepoLoadConfig) -> str:
    raw = f"{cfg.owner}/{cfg.repo}@{cfg.ref}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]


def _cache_paths(cfg: RepoLoadConfig) -> tuple[Path, Path]:
    cfg.cache_dir.mkdir(parents=True, exist_ok=True)
    key = _cache_key(cfg)
    return cfg.cache_dir / f"{key}.tar.gz", cfg.cache_dir / f"{key}.meta.json"


def _load_meta(meta_path: Path) -> dict:
    if not meta_path.exists():
        return {}
    try:
        return json.loads(meta_path.read_text("utf-8"))
    except Exception:
        return {}


def _save_meta(meta_path: Path, meta: dict) -> None:
    meta_path.write_text(json.dumps(meta, indent=2, sort_keys=True), "utf-8")


def _extract_tarball_to_memory(blob: bytes, cfg: RepoLoadConfig) -> Dict[str, str]:
    out: Dict[str, str] = {}
    tf = tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz")

    for member in tf.getmembers():
        if not member.isfile():
            continue
        if member.size > cfg.max_file_bytes:
            continue

        raw_path = member.name.replace("\\", "/")
        parts = raw_path.split("/", 1)
        if len(parts) != 2:
            continue
        path = parts[1]  # strip tar prefix

        if cfg.include_extensions is not None:
            if not any(path.lower().endswith(ext.lower()) for ext in cfg.include_extensions):
                continue

        if _matches_any_glob(path, cfg.deny_globs):
            continue
        if _matches_any_regex(path, cfg.deny_regex):
            continue

        f = tf.extractfile(member)
        if f is None:
            continue
        data = f.read()

        if _is_probably_binary(data):
            continue

        out[path] = data.decode("utf-8", errors="replace")

    return out


def load_repo_files_on_start(cfg: RepoLoadConfig) -> Dict[str, str]:
    """
    Startup-safe:
    - uses If-None-Match with ETag when available
    - caches tarball + meta
    - falls back to cached tarball if network/rate-limit fails (optional)
    """
    tar_path, meta_path = _cache_paths(cfg)
    meta = _load_meta(meta_path)

    now = time.time()
    fresh_enough = tar_path.exists() and (now - tar_path.stat().st_mtime) < cfg.ttl_s

    # If cache is fresh, avoid network entirely
    if fresh_enough:
        return _extract_tarball_to_memory(tar_path.read_bytes(), cfg)

    url = f"https://api.github.com/repos/{cfg.owner}/{cfg.repo}/tarball/{cfg.ref}"
    headers = _headers(cfg.token)

    # Conditional request if we have ETag
    if meta.get("etag"):
        headers["If-None-Match"] = meta["etag"]

    try:
        resp = requests.get(url, headers=headers, timeout=cfg.timeout_s)
        if resp.status_code == 304 and tar_path.exists():
            # Not modified; use cached tarball
            return _extract_tarball_to_memory(tar_path.read_bytes(), cfg)

        resp.raise_for_status()

        # Save tarball + meta
        tar_path.write_bytes(resp.content)
        meta = {
            "etag": resp.headers.get("ETag"),
            "fetched_at": int(time.time()),
            "owner": cfg.owner,
            "repo": cfg.repo,
            "ref": cfg.ref,
        }
        _save_meta(meta_path, meta)

        return _extract_tarball_to_memory(resp.content, cfg)

    except Exception as e:
        if cfg.allow_stale_on_error and tar_path.exists():
            # Rate-limited/offline? Still boot with last known good cache.
            return _extract_tarball_to_memory(tar_path.read_bytes(), cfg)
        raise


# Example for your repo
if __name__ == "__main__":
    cfg = RepoLoadConfig(
        owner="mifunedev",
        repo="workspace",
        ref="main",
        token=os.getenv("GITHUB_TOKEN"),  # strongly recommended for startup stability
        deny_globs=(

            "**/.github/**",
            "**/*.png",
            "**/*.pdf",
        ),
        include_extensions=(".md",),  # likely what you want for system prompt context
        ttl_s=60 * 60,  # check at most hourly
    )

    files = load_repo_files_on_start(cfg)
    print("Loaded", len(files), "files:", list(files)[:10])
    print(json.dumps(files, indent=4))


Loaded 11 files: ['.ruska/agents/command-builder.md', '.ruska/agents/skill-builder.md', '.ruska/skills/rlm/README.md', '.ruska/skills/rlm/SKILL.md', '.ruska/skills/rlm/references/prompt-templates.md', 'AGENTS.md', 'IDENTITY.md', 'MEMORY.md', 'SOUL.md', 'TOOLS.md']
{
    ".ruska/agents/command-builder.md": "---\nname: command-builder\ndescription: |\n  Elite command/skill builder for creating Claude Code custom commands.\n  MUST BE USED when user requests creating a new command, building a skill,\n  or designing workflow automation. Use when discussing command patterns or\n  slash commands for Claude Code.\ntools: Read, Glob, Grep, Edit, Write, Bash\nmodel: sonnet\n---\n\n# Command Builder Agent\n\nYou are an elite command builder for the Orchestra application. Your role is to create well-structured, actionable Claude Code commands (skills) that automate workflows, follow established patterns, and integrate seamlessly with the development process.\n\n## Your Expertise\n\nYou excel at:\n